In [0]:
spark.sql("USE CATALOG azdb_proyecto_mario")
spark.sql("USE SCHEMA lakeouse_proyecto")

df_clientes  = spark.table("azdb_proyecto_mario.lakeouse_proyecto.delta_clientes")
df_productos = spark.table("azdb_proyecto_mario.lakeouse_proyecto.delta_productos")
df_ventas    = spark.table("azdb_proyecto_mario.lakeouse_proyecto.delta_ventas")

df_ventas.show(5)



+-------+----------+---------+----------+--------+--------------+------------+
|IdVenta|     Fecha|IdCliente|IdProducto|Cantidad|PrecioUnitario|ImporteTotal|
+-------+----------+---------+----------+--------+--------------+------------+
|      1|2025-01-05|        1|         1|       1|        800.00|      800.00|
|      2|2025-01-10|        2|         3|       1|        150.00|      150.00|
|      3|2025-01-12|        3|         2|       2|         25.00|       50.00|
|      4|2025-01-02|        2|         2|       2|         21.50|       43.00|
|      5|2025-01-03|        3|         3|       3|         23.00|       69.00|
+-------+----------+---------+----------+--------+--------------+------------+
only showing top 5 rows


In [0]:
# ==========================
#   IMPORTS NECESARIOS
# ==========================
# Funciones de Spark SQL para crear agregaciones y columnas
from pyspark.sql.functions import (
    sum as _sum, 
    avg, 
    count, 
    countDistinct, 
    max, 
    datediff, 
    current_date
)

# Componentes para Machine Learning en Spark MLlib
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression


# =====================================================================
# 1. CREACIÓN DEL DATASET DE FEATURES POR CLIENTE
# =====================================================================
# A partir del dataframe df_ventas creamos variables que describen el
# comportamiento de cada cliente. Estas variables servirán como entrada
# (features) para el modelo predictivo.

df_features = (
    df_ventas.groupBy("IdCliente")
    .agg(
        # Número de compras realizadas por el cliente
        count("*").alias("NumCompras"),

        # Número de productos diferentes que compró
        countDistinct("IdProducto").alias("NumProductosDistintos"),

        # Cantidad total de unidades compradas
        _sum("Cantidad").alias("CantidadTotal"),

        # Precio medio pagado por producto
        avg("PrecioUnitario").alias("PrecioMedio"),

        # Importe medio por compra
        avg("ImporteTotal").alias("TicketMedio"),

        # Importe total gastado → será nuestro "label"
        _sum("ImporteTotal").alias("ImporteTotalCliente"),

        # Fecha de la última compra
        max("Fecha").alias("UltimaCompra")
    )
)

# Añadimos una nueva feature:
# "RecenciaDias" = días desde la última compra hasta hoy.
# Muy usado en modelos de análisis RFM y predicción de gasto.
df_features = df_features.withColumn(
    "RecenciaDias",
    datediff(current_date(), df_features["UltimaCompra"])
)


# =====================================================================
# 2. PREPARACIÓN DE DATOS PARA ENTRENAR EL MODELO
# =====================================================================

# La columna a predecir (variable objetivo) se llamará "label".
df_ml = df_features.withColumnRenamed("ImporteTotalCliente", "label")

# Seleccionamos las columnas que serán las FEATURES del modelo.
input_cols = [
    "NumCompras",
    "NumProductosDistintos",
    "CantidadTotal",
    "PrecioMedio",
    "TicketMedio",
    "RecenciaDias"
]

# VectorAssembler convierte varias columnas numéricas en un único vector,
# que es el formato requerido por Spark ML.
assembler = VectorAssembler(inputCols=input_cols, outputCol="features")

# Aplicamos el assembler al dataset.
df_ml = assembler.transform(df_ml)


# =====================================================================
# 3. DIVISIÓN EN TRAIN / TEST
# =====================================================================
# Se divide el dataset en un conjunto de entrenamiento (80%)
# y un conjunto de test (20%) para evaluar el modelo.
train, test = df_ml.randomSplit([0.8, 0.2], seed=42)


# =====================================================================
# 4. ENTRENAMIENTO DEL MODELO (REGRESIÓN LINEAL)
# =====================================================================
# Creamos el modelo especificando qué columnas usar.
lr = LinearRegression(featuresCol="features", labelCol="label")

# Entrenamos el modelo con los datos de entrenamiento.
model = lr.fit(train)


# =====================================================================
# 5. GENERAR PREDICCIONES
# =====================================================================
# Aplicamos el modelo al conjunto de test.
preds = model.transform(test)

# Mostramos por cada cliente el valor real (label)
# vs. el valor predicho por el modelo (prediction).
preds.select("IdCliente", "label", "prediction").show()


+---------+------+------------------+
|IdCliente| label|        prediction|
+---------+------+------------------+
|        3|119.00|119.99247484723418|
|        7| 58.00| 57.99999828437586|
|        9|128.00|127.99999856269221|
|       14|158.00|157.99999865570874|
|       20|242.50| 242.4999986673132|
|       24|218.00|217.99999884174179|
|       30|317.50|317.49999868367706|
|       36| 72.50| 72.50000017943944|
|       46| 87.50| 87.50000087448029|
|       47|178.00| 178.0000003858621|
|       48|271.50| 271.4999998633101|
|       50|467.50|467.49999871640455|
|       52|193.00|193.00000064854794|
|       56|102.50|102.50000156952115|
|       63|339.00| 339.0000003968636|
|       70|617.50|  617.499998749132|
|       78|406.50|406.50000093041683|
+---------+------+------------------+



In [0]:
import mlflow
from mlflow.models import infer_signature

mlflow.set_registry_uri("databricks-uc")

registered_name = "azdb_proyecto_mario.lakeouse_proyecto.modelo_ventas_clientes"

signature = infer_signature(
    train.select("features").toPandas(),
    preds.select("prediction").toPandas()
)

# Replace with your actual Unity Catalog volume path
dfs_tmpdir = "/Volumes/azdb_proyecto_mario/lakeouse_proyecto/volumen_ml/mlflow_tmp"

with mlflow.start_run(run_name="demo_ventas_por_cliente"):
    mlflow.log_param("algorithm", "LinearRegression")
    rmse = model.summary.rootMeanSquaredError
    mlflow.log_metric("rmse", rmse)
    mlflow.spark.log_model(
        model,
        artifact_path="model",
        registered_model_name=registered_name,
        signature=signature,
        dfs_tmpdir=dfs_tmpdir
    )

2025/12/03 20:12:23 WARNING mlflow.models.signature: Failed to infer schema for inputs. Setting schema to `Schema([ColSpec(type=AnyType())]` as default. Note that MLflow doesn't validate data types during inference for AnyType. To see the full traceback, set logging level to DEBUG.
2025/12/03 20:12:28 WARNING mlflow.utils.requirements_utils: Found pyspark version (4.0.0+databricks.connect.17.2.2) contains a local version label (+databricks.connect.17.2.2). MLflow logged a pip requirement for this package as 'pyspark==4.0.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2025/12/03 20:12:32 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /local_disk0/user_tmp_data/spark-81e11a29-7f5c-40a1-9d3b-c7/tmprkd6nirf/model, flavor: spark). Fall back to return ['pyspark==4.0.0']. Set logging level to DEBUG to s